# Детекция ботов по поведенческим данным

## Подход к решению

Задача заключается в бинарной классификации cookie: необходимо определить, относится ли пользователь к ботам или к обычным пользователям.

Основная идея решения — представить последовательность событий каждой cookie в виде набора агрегированных поведенческих признаков и обучить на них модель машинного обучения.

### 1. Подготовка данных

Для каждой cookie используются события, попавшие в соответствующее временное окно. События очищаются и сортируются по времени, чтобы сохранить последовательность действий пользователя.

### 2. Feature Engineering

Из событий строятся группы признаков, описывающие различные аспекты поведения:

- **Активность:** количество событий и распределение типов событий.
- **Разнообразие:** число уникальных объявлений, категорий, локаций, поисковых запросов и других объектов.
- **Повторяемость:** доля повторных объявлений и поисковых запросов.
- **Поиск:** глубина просмотра страниц, последовательность страниц, характеристики поисковых запросов.
- **Временное поведение:** интервалы между событиями, их средние значения, квантили, доля очень коротких и длинных интервалов.
- **Сессии:** количество сессий, их длительность и количество событий внутри сессий.
- **Временная активность:** распределение активности по часам и плотность событий.
- **Pointer:** наличие координат pointer, разнообразие позиций, распределение координат и характеристики перемещения.
- **Platform / User-Agent:** платформа, характеристики браузера и признаки автоматизированного окружения.
- **Последовательности событий:** частоты характерных переходов между типами событий.
- **Поведенческие отношения:** например, количество просмотров на один поиск или количество действий после просмотра объявления.
- **Popularity features:** насколько часто связанные с cookie объявления и поисковые запросы встречаются у других пользователей.
- **Meta-признаки:** возраст cookie относительно временного окна и характеристики времени создания.

Таким образом, исходная последовательность событий преобразуется в табличное представление: одна строка соответствует одной cookie, а столбцы — её поведенческим признакам.

### 3. Модель

Для классификации используется `HistGradientBoostingClassifier` из `scikit-learn`.

Используется несколько моделей с разными `random_state`, после чего их вероятности принадлежности к классу бот усредняются.

### 4. Валидация

Так как данные имеют временную структуру, используется **time-based validation**.

Для каждого validation-периода модель обучается только на данных, которые произошли до начала этого периода. Будущие относительно validation-периода данные в обучении не используются.

Это позволяет приблизить валидацию к реальному сценарию: модель обучается на прошлом и делает предсказания для будущего.

### 5. Метрики

Основная метрика задачи:

**P@Recall ≥ 0.7**

Дополнительно рассчитываются:

- **PR-AUC** — качество ранжирования положительного класса;
- **ROC-AUC** — общая способность модели разделять два класса.

Для итогового решения модель обучается на доступных обучающих данных и формирует вероятность принадлежности каждой cookie к классу ботов.

## 1. Загрузка и очистка

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from metric import precision_at_recall
import warnings
warnings.filterwarnings('ignore')

In [2]:
def load():
    train = pd.read_csv('data/train.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
    test = pd.read_csv('data/test.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
    events = pd.read_csv('data/events.csv.gz', parse_dates=['event_ts'])
    return train, test, events

In [3]:
def clean_events(ev, meta):
    ev = ev.drop_duplicates()
    ev = ev.merge(meta[['cookie_id', 'window_start_ts', 'window_end_ts']], on='cookie_id')
    ev = ev[(ev.event_ts >= ev.window_start_ts) & (ev.event_ts < ev.window_end_ts)].copy()
    ev['plat'] = ev.platform.str.lower().replace({'iphone': 'ios'})
    ev['plat'] = ev.plat.replace({'desktop': 'web'})
    ev = ev.sort_values(['cookie_id', 'event_ts'], kind='mergesort').reset_index(drop=True)
    ev['seconds_from_strt'] = (ev.event_ts - ev.window_start_ts).dt.total_seconds()
    return ev

## 2. Признаки

In [4]:
def build_features(meta, ev, train_meta, is_train):
    raw_ev = ev
    ev = clean_events(raw_ev, meta)
    g = ev.groupby('cookie_id')
    F = pd.DataFrame(index=meta.cookie_id)
    F['n'] = g.size()
    F['log_n'] = np.log1p(F.n)

    #считаем количество и долю каждого события 
    for e in sorted(ev.event_name.dropna().unique()):
        c = ev[ev.event_name == e].groupby('cookie_id').size()
        F['n_' + e] = c
        F['r_' + e] = c / F.n
    F = F.fillna(0)

    # количество и доли уникальных значений в разных колонках
    for col in ['item_id', 'item_category', 'item_location', 'seller_type', 'search_query', 'search_page', 'user_agent', 'platform', 'plat', 'eid']:
        F['nu_' + col] = g[col].nunique()

    F['item_uniq_ratio'] = F.nu_item_id / (ev[ev.item_id.notna()].groupby('cookie_id').size().reindex(F.index).fillna(np.nan))
    F['q_uniq_ratio'] = F.nu_search_query / (ev[ev.search_query.notna()].groupby('cookie_id').size().reindex(F.index))
    F['top_item_share'] = g.item_id.apply(lambda s: s.value_counts(normalize=True).iloc[0] if s.notna().any() else 0)
    F['top_query_share'] = g.search_query.apply(lambda s: s.value_counts(normalize=True).iloc[0] if s.notna().any() else 0)
    
    
    # признаки по номерам поисковых страниц
    sp = ev[ev.search_page.notna()].groupby('cookie_id').search_page
    F['sp_max'] = sp.max(); F['sp_mean'] = sp.mean(); F['sp_std'] = sp.std()
    F['sp_frac_gt1'] = sp.apply(lambda s: (s > 1).mean())
    F['sp_frac_gt3'] = sp.apply(lambda s: (s > 3).mean())
    
    # насколько последовательно происходило перемешение по страницам поиска
    def seq_frac(s):
        v = s.values
        return np.mean(np.diff(v) == 1) if len(v) > 1 else np.nan
    F['sp_seq_frac'] = sp.apply(seq_frac)
    
    
    # признаки по id объявления
    it = ev[ev.item_id.notna()].groupby('cookie_id').item_id
    F['item_min'] = it.min()
    F['item_mean'] = it.mean()
    F['item_std'] = it.std()
    F['item_seq_frac'] = it.apply(lambda s: np.mean(np.abs(np.diff(s.values)) == 1) if len(s) > 1 else np.nan)
    
    
    # признаки по времени между соседними событиями
    ev['dt'] = ev.groupby('cookie_id').seconds_from_strt.diff()
    d = ev[ev.dt.notna()].groupby('cookie_id').dt
    F['dt_mean'] = d.mean()
    F['dt_med'] = d.median()
    F['dt_std'] = d.std()
    F['dt_min'] = d.min()
    F['dt_max'] = d.max()
    F['dt_cv'] = F.dt_std / F.dt_mean
    F['dt_q10'] = d.quantile(.1); F['dt_q90'] = d.quantile(.9)
    F['dt_zero_frac'] = d.apply(lambda s: (s == 0).mean())
    F['dt_le1_frac'] = d.apply(lambda s: (s <= 1).mean())
    F['dt_le3_frac'] = d.apply(lambda s: (s <= 3).mean())
    F['dt_ge600_frac'] = d.apply(lambda s: (s >= 600).mean())
    F['dt_mode_share'] = d.apply(lambda s: s.value_counts(normalize=True).iloc[0])
    F['dt_nuniq_ratio'] = d.nunique() / d.size()
    F['dt_logstd'] = d.apply(lambda s: np.log1p(s).std())
    F['dt_lag1_corr'] = d.apply(lambda s: np.corrcoef(s.values[:-1], s.values[1:])[0, 1] if len(s) > 4 and s.std() > 0 else np.nan)
    
    
    # время начала конца длительность темп 
    tg = ev.groupby('cookie_id').seconds_from_strt
    F['t_first'] = tg.min()
    F['t_last'] = tg.max()
    F['span'] = F.t_last - F.t_first
    F['rate'] = F.n / (F.span + 1)
    
    
    # признаки по кол-ву часов и в какие часы активность макс событий за минуту/час доля повторяющихся ts 
    ev['hour'] = ev.event_ts.dt.hour
    hg = ev.groupby('cookie_id').hour
    F['n_hours'] = hg.nunique()
    F['night_frac'] = hg.apply(lambda s: ((s < 6)).mean())
    F['hour_mean'] = hg.mean()
    F['hour_std'] = hg.std()
    F['sec_nuniq'] = ev.groupby('cookie_id').event_ts.apply(lambda s: s.dt.second.nunique())
    F['max_per_min'] = ev.groupby(['cookie_id', ev.event_ts.dt.floor('min')]).size().groupby('cookie_id').max()
    F['max_per_hour'] = ev.groupby(['cookie_id', ev.event_ts.dt.floor('h')]).size().groupby('cookie_id').max()
    F['dup_ts_frac'] = 1 - g.event_ts.nunique() / F.n
    
    
    # кол-во, продолжительность сессий (>30мин) + кол-во событий в сессии
    ev['newsess'] = (ev.dt.isna() | (ev.dt > 1800)).astype(int)
    F['n_sess'] = ev.groupby('cookie_id').newsess.sum()
    ev['sid'] = ev.groupby('cookie_id').newsess.cumsum()
    ss = ev.groupby(['cookie_id', 'sid'])
    sl = ss.size().groupby('cookie_id') 
    F['sess_len_mean'] = sl.mean()
    F['sess_len_max'] = sl.max()
    sd = (ss.seconds_from_strt.max() - ss.seconds_from_strt.min()).groupby('cookie_id')
    F['sess_dur_mean'] = sd.mean()
    F['sess_dur_max'] = sd.max()
    
    
    # работаем с данными курсора, смотрим на долю событий где он есть
    pm = ev.pointer_x.notna()
    F['ptr_frac'] = pm.groupby(ev.cookie_id).mean()
    #кол-во событий с курсором, кол-во и доля уникальных
    pe = ev[pm]
    pg = pe.groupby('cookie_id')
    F['ptr_n'] = pg.size()
    F['ptr_nuniq'] = pg.apply(lambda x: len(set(zip(x.pointer_x, x.pointer_y))))
    F['ptr_uniq_ratio'] = F.ptr_nuniq / F.ptr_n
    #среднее и разброс x y
    F['ptr_x_mean'] = pg.pointer_x.mean()
    F['ptr_y_mean'] = pg.pointer_y.mean()
    F['ptr_x_std'] = pg.pointer_x.std()
    F['ptr_y_std'] = pg.pointer_y.std()
    #как часто находится в (0, 0) или на краю экрана
    F['ptr_zero_frac'] = pg.apply(lambda x: ((x.pointer_x == 0) & (x.pointer_y == 0)).mean())
    F['ptr_edge_frac'] = pg.apply(lambda x: ((x.pointer_x == 0) | (x.pointer_x == 1920) | (x.pointer_y == 0) | (x.pointer_y == 1080)).mean())
    #доля кратных 10
    F['ptr_x_mod10'] = pg.pointer_x.apply(lambda s: (s % 10 == 0).mean())
    #доля web имеющих курсор и доля мобильных не имеющих курсор
    F['ptr_frac_by_plat_web'] = ev[ev.plat == 'web'].groupby('cookie_id').pointer_x.apply(lambda s: s.notna().mean())
    F['ptr_mobile'] = ev[ev.plat != 'web'].groupby('cookie_id').pointer_x.apply(lambda s: s.notna().mean())
    #расстояние между соседними координатами
    def ptr_step(x):
        dx = np.hypot(np.diff(x.pointer_x.values), np.diff(x.pointer_y.values))
        return dx.mean() if len(dx) else np.nan
    F['ptr_step_mean'] = pg.apply(ptr_step)
    
    
    #доля и кол-во событий с разных платформ
    for p in ['web', 'android', 'ios']:
        F['plat_' + p] = (ev.plat == p).groupby(ev.cookie_id).mean()
    F['plat_nu'] = g.platform.nunique()


    #работаем с user agent проверяем долю отсутствия ua
    ua = ev.user_agent.fillna('')
    F['ua_null'] = ev.user_agent.isna().groupby(ev.cookie_id).mean()
    for name, pat in [('headless', 'Headless'), ('linux', 'X11|Linux x86'), ('win', 'Windows'), ('mac', 'Macintosh'), ('yandex', 'YaBrowser'),
                      ('firefox', 'Firefox'), ('android', 'Android'), ('iphone', 'iPhone|iPad'), ('mobile', 'Mobile'), ('python', 'python|curl|requests|bot|spider|crawl|scrapy|Playwright|Puppeteer|Selenium|phantom', )]:
        F['ua_' + name] = ua.str.contains(pat, case=False, regex=True).groupby(ev.cookie_id).mean()
    ver = ua.str.extract(r'Chrome/(\d+)')[0].astype(float)
    F['chrome_ver'] = ver.groupby(ev.cookie_id).mean()
    F['chrome_ver_min'] = ver.groupby(ev.cookie_id).min()
    fv = ua.str.extract(r'Firefox/(\d+)')[0].astype(float)
    F['ff_ver'] = fv.groupby(ev.cookie_id).mean()
    F['ua_len'] = ua.str.len().groupby(ev.cookie_id).mean()
    
    
    # платформа и user agent не совпадают
    mism = ((ev.plat == 'web') & ua.str.contains('Android|iPhone', regex=True)) | ((ev.plat == 'android') & ~ua.str.contains('Android')) | ((ev.plat == 'ios') & ~ua.str.contains('iPhone|iPad|Mac'))
    F['ua_plat_mismatch'] = mism.groupby(ev.cookie_id).mean()
    
    
    # насколько часто встречается user agent, популярность ua для cookie
    uac = ev.groupby('user_agent').cookie_id.nunique()
    ev['ua_pop'] = ev.user_agent.map(uac)
    F['ua_pop_mean'] = ev.groupby('cookie_id').ua_pop.mean()
    
    
    # последовательность действий
    ev['prev'] = ev.groupby('cookie_id').event_name.shift()
    transitions = ev[ev.prev.notna()]
    act = (transitions.prev + '>' + transitions.event_name)
    # примеры нормального поведения пользователя
    top = ['item_view>item_view', 'search_results_view>search_results_view', 'search_results_view>item_view', 'item_view>photo_swipe',
           'item_view>seller_page_view', 'item_view>search_results_view', 'photo_swipe>photo_swipe', 'seller_page_view>item_view',
           'item_view>contact_phone_show', 'item_view>favorite_add', 'seller_page_view>seller_page_view', 'seller_page_view>search_results_view']
    n_transitions = act.groupby(transitions.cookie_id).size()
    for k in top:
        F['act_' + k] = ((act == k).groupby(transitions.cookie_id).sum() / n_transitions)
    
    
    # соотношения разных действий к друг другу
    F['view_per_search'] = F.n_item_view / (F.n_search_results_view + 1)
    F['contact_total'] = F.n_contact_phone_show + F.n_contact_chat_open + F.n_contact_message_sent
    F['contact_per_view'] = F.contact_total / (F.n_item_view + 1)
    F['swipe_per_view'] = F.n_photo_swipe / (F.n_item_view + 1)
    F['fav_per_view'] = F.n_favorite_add / (F.n_item_view + 1)
    F['seller_per_view'] = F.n_seller_page_view / (F.n_item_view + 1)
    F['login_any'] = (F.n_login > 0).astype(int)
    
    
    # средняя длина и кол-во слов в поисковом запросе
    q = ev[ev.search_query.notna()]
    F['q_len'] = q.groupby('cookie_id').search_query.apply(lambda s: s.str.len().mean())
    F['q_words'] = q.groupby('cookie_id').search_query.apply(lambda s: s.str.split().str.len().mean())
    
    
    # доля опроделенного типа продовца
    for st in ev.seller_type.dropna().unique():
        F['st_' + st] = (ev.seller_type == st).groupby(ev.cookie_id).sum() / ev.seller_type.notna().groupby(ev.cookie_id).sum().replace(0, np.nan)
    
    
    # самая популярная категория и локация по cookie
    F['top_cat_share'] = g.item_category.apply(lambda s: s.value_counts(normalize=True).iloc[0] if s.notna().any() else 0)
    F['top_loc_share'] = g.item_location.apply(lambda s: s.value_counts(normalize=True).iloc[0] if s.notna().any() else 0)
    
    
    # meta и работа с временным окном
    m = meta.set_index('cookie_id')
    F['age_days'] = (m.window_start_ts - m.cookie_created_at).dt.total_seconds() / 86400
    F['created_in_window'] = (m.cookie_created_at >= m.window_start_ts).astype(int)
    F['created_hour'] = m.cookie_created_at.dt.hour
    F['dow'] = m.window_start_ts.dt.dayofweek


    #признаки курсора только на web
    web = ev[ev.plat == 'web']
    out = {}
    # доля веб событий с курсором и их количество
    out['web_ptr_presence'] = web.groupby('cookie_id').pointer_x.apply(lambda s: s.notna().mean())
    out['n_web'] = web.groupby('cookie_id').size()
    web_ptr = web[web.pointer_x.notna()]
    wpg = web_ptr.groupby('cookie_id')
    out['n_ptr_web'] = wpg.size()
    

    # по каждой оси посчитаем медиану мин макс размах отличие от равномерного распределения по экрану прилипание к границам
    for c, W in [('pointer_x', 1920), ('pointer_y', 1080)]:
        out[c + '_med'] = wpg[c].median()
        out[c + '_min'] = wpg[c].min()
        out[c + '_max'] = wpg[c].max()
        out[c + '_iqr'] = wpg[c].quantile(.75) - wpg[c].quantile(.25)
    

    #расстояние до типичного центра курсора ботов
    dc = np.hypot((web_ptr.pointer_x - 680) / 360, (web_ptr.pointer_y - 500) / 260)
    dcg = dc.groupby(web_ptr.cookie_id)
    out['ptr_dc_mean'] = dcg.mean()
    out['ptr_dc_min'] = dcg.min()
    out['ptr_center_frac'] = dcg.apply(lambda s: (s < 1.5).mean())
    out['ptr_corr'] = wpg.apply(lambda x: np.corrcoef(x.pointer_x, x.pointer_y)[0, 1] if len(x) > 4 else np.nan)
    
    
    #доля событий с курсором внутри каждого типа события если веб
    for en in ['item_view', 'search_results_view', 'seller_page_view', 'photo_swipe']:
        w = web[web.event_name == en]
        out['wptr_' + en] = w.groupby('cookie_id').pointer_x.apply(lambda s: s.notna().mean())

    #популярность объявлений и запросов
    ref = clean_events(raw_ev, train_meta)
    pop_item = ref.groupby('item_id').cookie_id.nunique()
    pop_query = ref.groupby('search_query').cookie_id.nunique()
    item_pop = ev.item_id.map(pop_item).fillna(0).where(ev.item_id.notna())
    query_pop = ev.search_query.map(pop_query).fillna(0).where(ev.search_query.notna())
    if is_train: 
        item_pop = item_pop - ev.item_id.notna().astype(int)
        query_pop = query_pop - ev.search_query.notna().astype(int)
    ipg = item_pop.groupby(ev.cookie_id)
    qpg = query_pop.groupby(ev.cookie_id)
    out['item_pop_mean'] = ipg.mean()
    out['item_pop_max'] = ipg.max()
    out['item_pop_min'] = ipg.min()
    out['q_pop_mean'] = qpg.mean()
    out['q_pop_max'] = qpg.max()
    out['q_pop_min'] = qpg.min()

    # число разных локаций на одно объявление
    out['loc_per_item'] = g.item_location.nunique() / (g.item_id.nunique() + 1)

    F = F.join(pd.DataFrame(out).reindex(F.index))
    return F.reset_index().replace([np.inf, -np.inf], np.nan)


## 3. Сбор признаков

In [5]:
train, test, events = load()

Ftrain = build_features(train, events, train_meta=train, is_train=True)
Ftest = build_features(test, events, train_meta=train, is_train=False)
assert (Ftrain.cookie_id.values == train.cookie_id.values).all() and (Ftest.cookie_id.values == test.cookie_id.values).all()
cols = [c for c in Ftrain.columns if c != 'cookie_id']
y = train.target.values
print('признаков:', len(cols), '| train:', Ftrain.shape, '| test:', Ftest.shape, '| доля ботов в train:', y.mean().round(4))

признаков: 167 | train: (11091, 168) | test: (4909, 168) | доля ботов в train: 0.0811


## 4. Валидация по времени

In [6]:
def make_model(seed):
    return HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, max_leaf_nodes=30, min_samples_leaf=20,
                                          l2_regularization=1.0, max_features=0.4, random_state=seed)

chunks = [('2026-04-10', '2026-04-12'), ('2026-04-13', '2026-04-15'), ('2026-04-16', '2026-04-19')]
d = train.window_start_ts
rows = []
for a, b in chunks:
    v = d.between(a, b).values
    t = d.lt(a).values
    p = np.mean([make_model(s).fit(Ftrain.loc[t, cols], y[t]).predict_proba(Ftrain.loc[v, cols])[:, 1] for s in range(2)], axis=0)
    rows.append({'валидация': f'{a} - {b}', 'n': int(v.sum()), 'доля ботов': round(y[v].mean(), 3),
                 'P@R≥0.7': round(precision_at_recall(y[v], p), 4),
                 'PR-AUC': round(average_precision_score(y[v], p), 4), 'ROC-AUC': round(roc_auc_score(y[v], p), 4)})
res = pd.DataFrame(rows)
print(res.to_string(index=False))
print('\nсреднее P@R≥0.7:', res['P@R≥0.7'].mean().round(4))

              валидация    n  доля ботов  P@R≥0.7  PR-AUC  ROC-AUC
2026-04-10 - 2026-04-12 2625       0.087   0.8205  0.7843   0.9222
2026-04-13 - 2026-04-15 2470       0.076   0.9110  0.8359   0.9499
2026-04-16 - 2026-04-19 2691       0.082   0.8579  0.8078   0.9517

среднее P@R≥0.7: 0.8631


## 5. Финальное обучение и submission

In [8]:
ps = [make_model(s).fit(Ftrain[cols], y).predict_proba(Ftest[cols])[:, 1] for s in range(10)]
sub = pd.DataFrame({'cookie_id': Ftest.cookie_id, 'score': np.mean(ps, axis=0)})

ss = pd.read_csv('sample_submission.csv')
assert len(sub) == len(test) and sub.cookie_id.is_unique and set(sub.cookie_id) == set(ss.cookie_id)
assert sub.score.between(0, 1).all() and sub.score.notna().all()
sub.to_csv('submission.csv', index=False)
print(sub.score.describe().round(4))

count    4909.0000
mean        0.0825
std         0.2169
min         0.0015
25%         0.0049
50%         0.0111
75%         0.0271
max         0.9874
Name: score, dtype: float64
